## Robust Log-Based Anomaly Detection on Unstable Log Data

### Обработка данных



In [4]:
import pandas as pd
import re
import csv
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

In [5]:
df = pd.read_csv("Event_traces.csv")
df.drop(['Type', 'TimeInterval', 'Latency'], axis=1, inplace=True)

def to_list(x):
    if isinstance(x, list):
        return x

    x = x.strip().strip("[]")

    if x == "":
        return []

    return [i.strip() for i in x.split(",")]

df["Features"] = df["Features"].apply(to_list)

Для тренировочные\тестовые данные разделены в соотношении 80 на 20

In [10]:
train, test = train_test_split(
    df, test_size=0.2, random_state=42
)

In [11]:
all_events = train["Features"].explode()

le = LabelEncoder()
le.fit(all_events)

df["events"] = df["Features"].apply(lambda seq: le.transform(seq).tolist())

In [14]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42
)

In [17]:
train_df["Label"] = train_df["Label"].map({
    "Success": 0,
    "Fail": 1
})

test_df["Label"] = test_df["Label"].map({
    "Success": 0,
    "Fail": 1
})

### Модель


Создаем DataLoader

In [18]:
class LogDataset(Dataset):
    def __init__(self, sequences, labels, max_len):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]

        return (
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(label, dtype=torch.float)
        )

In [19]:
train_dataset = LogDataset(
    train_df["events"].tolist(),
    train_df["Label"].tolist(),
    298
)

test_dataset = LogDataset(
    test_df["events"].tolist(),
    test_df["Label"].tolist(),
    298
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Реализуем модель

In [20]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # x: (batch, seq_len, hidden*2)
        weights = self.attn(x).squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(weights, dim=1)

        context = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return context, weights


class LogRobust(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(hidden_dim)

        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        emb = self.embedding(x)

        lstm_out, _ = self.lstm(emb)

        context, attn = self.attention(lstm_out)

        out = self.fc(context)

        return torch.sigmoid(out), attn

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred, _ = model(x)

        pred = pred.squeeze()

        loss = criterion(pred, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            pred, _ = model(x)
            pred = pred.squeeze()

            preds.extend(pred.cpu().tolist())
            labels.extend(y.cpu().tolist())

    return preds, labels

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LogRobust(vocab_size=298).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

Сделали всего 5 эпох, на всех эпохах Loss падает, модель обучается

In [24]:
epochs = 5

for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader)

    print(f"Эпоха {epoch+1} из {epochs}, Loss: {loss:.4f}")

Эпоха 1 из 5, Loss: 0.0059
Эпоха 2 из 5, Loss: 0.0012
Эпоха 3 из 5, Loss: 0.0011
Эпоха 4 из 5, Loss: 0.0009
Эпоха 5 из 5, Loss: 0.0008


In [25]:
preds, labels = evaluate(model, test_loader)

y_pred = [1 if p > 0.5 else 0 for p in preds]

Считаем метрику:

In [34]:
print('Метрика F1:', f1_score(test_df['Label'], y_pred))

Метрика F1: 0.9967134747535106
